In [36]:
# ============================================================
# 4 MODELS × 1 SEED × 100 EPOCHS
# Train locally in Colab -> upload complete result to Drive
# ============================================================

from google.colab import auth
auth.authenticate_user()

import os
import json
import time
import mimetypes
import subprocess
from pathlib import Path

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload


# ============================================================
# SETTINGS
# ============================================================

ROOT = Path("/content/durian_training")
CONTROL = ROOT / "experiment/controlled_experiment"
CONFIGS = CONTROL / "configs"
OUTPUTS = CONTROL / "outputs"
WEIGHTS = CONTROL / "weights"
RUNNER = ROOT / "experiment/training_model/candidate_runner.py"

SEED = 20260917

ENV_YOLO = Path("/content/env_yolo/bin/python")
ENV_RF = Path("/content/env_rfdetr/bin/python")
ENV_DFINE = Path("/content/env_dfine/bin/python")

DFINE_REPO = Path("/content/D-FINE")

EXPECTED_COMMIT = "6db81100a857b1e45e58d73872e7c89cb3766c30"


# Google Drive result folder IDs
DRIVE_RESULT_FOLDERS = {
    "yolo26n": "1mg7gjC7lRcUW05dF_5lWF0vfvuqv3ubX",
    "yolo26n_p2": "1J0BT4Q-cHF1Ow-P8Q8xpjeOFgekQb-RS",
    "rfdetr_n": "1R8CpLk0jTG4o5h8HpNGnZeowi1CSmrzm",
    "dfine_n": "1Trv4a-1lSFkHsDomfKfiJyeFk70y-My_",
}

RUNS = [
    ("yolo26n", ENV_YOLO),
    ("yolo26n_p2", ENV_YOLO),
    ("rfdetr_n", ENV_RF),
    ("dfine_n", ENV_DFINE),
]

drive_service = build("drive", "v3")


# ============================================================
# DRIVE HELPERS
# ============================================================

def drive_children(folder_id):
    result = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id,name,mimeType)",
        pageSize=1000,
    ).execute()

    return result.get("files", [])


def create_drive_folder(name, parent_id):
    metadata = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }

    item = drive_service.files().create(
        body=metadata,
        fields="id"
    ).execute()

    return item["id"]


def upload_file(local_file, parent_id):
    local_file = Path(local_file)

    mime_type, _ = mimetypes.guess_type(str(local_file))
    mime_type = mime_type or "application/octet-stream"

    metadata = {
        "name": local_file.name,
        "parents": [parent_id],
    }

    media = MediaFileUpload(
        str(local_file),
        mimetype=mime_type,
        resumable=True,
        chunksize=10 * 1024 * 1024,
    )

    request = drive_service.files().create(
        body=metadata,
        media_body=media,
        fields="id,name",
    )

    response = None

    while response is None:
        status, response = request.next_chunk()

        if status:
            pct = int(status.progress() * 100)
            print(f"      upload {local_file.name}: {pct}%")

    return response


def upload_directory_contents(local_dir, drive_folder_id):
    """
    Upload CONTENTS of local_dir directly into existing Drive result folder.
    Does NOT create seed_* or trial_* wrapper folder.
    """

    local_dir = Path(local_dir)

    assert local_dir.is_dir(), f"Missing local result: {local_dir}"

    def recursive_upload(src_dir, dst_drive_id):

        for item in sorted(src_dir.iterdir()):

            if item.is_dir():

                print(f"   📁 {item.relative_to(local_dir)}")

                new_drive_folder = create_drive_folder(
                    item.name,
                    dst_drive_id
                )

                recursive_upload(
                    item,
                    new_drive_folder
                )

            elif item.is_file():

                print(f"   ↑ {item.relative_to(local_dir)}")

                upload_file(
                    item,
                    dst_drive_id
                )

    recursive_upload(
        local_dir,
        drive_folder_id
    )


# ============================================================
# PRECHECK
# ============================================================

print("=" * 90)
print("FINAL 4-MODEL / 1-SEED PRECHECK")
print("=" * 90)

commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"],
    text=True
).strip()

assert commit == EXPECTED_COMMIT, (
    f"Wrong source commit\n"
    f"Expected: {EXPECTED_COMMIT}\n"
    f"Actual:   {commit}"
)

print("Git commit: PASS")


for p in [
    ENV_YOLO,
    ENV_RF,
    ENV_DFINE,
    RUNNER,
]:
    assert p.exists(), f"Missing: {p}"


for candidate, _ in RUNS:

    cfg = json.loads(
        (CONFIGS / f"{candidate}.json").read_text()
    )

    assert cfg["training"]["epochs"] == 100
    assert cfg["training"]["early_stopping"]["enabled"] is False
    assert SEED in cfg["training"]["seeds"]

    print(
        f"{candidate:12s} -> "
        f"100 epochs | seed {SEED}"
    )


# D-FINE pinned source check
dfine_commit = subprocess.check_output(
    ["git", "-C", str(DFINE_REPO), "rev-parse", "HEAD"],
    text=True
).strip()

assert (
    dfine_commit
    == "956d1709314c2c6a4df6f34de232054578a7449f"
)

print("D-FINE commit: PASS")


# Drive result folders must currently be empty
for model, folder_id in DRIVE_RESULT_FOLDERS.items():

    children = drive_children(folder_id)

    assert len(children) == 0, (
        f"\nDrive folder for {model} is NOT empty.\n"
        f"Refusing to mix/overwrite experiment results.\n"
        f"Existing: {[x['name'] for x in children]}"
    )

    print(f"Drive {model}/result: EMPTY / READY")


print("\nPRECHECK: PASS")
print("=" * 90)


# ============================================================
# TRAIN + UPLOAD
# ============================================================

for run_no, (candidate, python_bin) in enumerate(
    RUNS,
    start=1
):

    print("\n")
    print("#" * 90)
    print(
        f"MODEL {run_no}/4"
        f" | {candidate}"
        f" | seed={SEED}"
        f" | 100 epochs"
    )
    print("#" * 90)


    local_run_dir = (
        OUTPUTS /
        candidate /
        f"seed_{SEED}"
    )


    # Prevent accidental overwrite
    if local_run_dir.exists():
        raise RuntimeError(
            f"\nLocal controlled run already exists:\n"
            f"{local_run_dir}\n\n"
            f"Not overwriting scientific result."
        )


    cmd = [
        str(python_bin),
        str(RUNNER),

        "--config",
        str(CONFIGS / f"{candidate}.json"),

        "--execute",

        "--seed",
        str(SEED),

        "--weight-dir",
        str(WEIGHTS),
    ]


    if candidate == "dfine_n":
        cmd += [
            "--dfine-repo",
            str(DFINE_REPO),
        ]


    print("\nTRAINING START")
    start = time.time()


    process = subprocess.run(
        cmd,
        cwd=ROOT,
        env={
            **os.environ,
            "PYTHONUNBUFFERED": "1",
            "MPLBACKEND": "Agg",
        }
    )


    elapsed = time.time() - start


    if process.returncode != 0:
        raise RuntimeError(
            f"{candidate} FAILED "
            f"(return code {process.returncode})"
        )


    assert local_run_dir.is_dir(), (
        f"Training finished but output folder missing:\n"
        f"{local_run_dir}"
    )


    checkpoints = sorted(
        list(local_run_dir.rglob("*.pt")) +
        list(local_run_dir.rglob("*.pth"))
    )


    assert checkpoints, (
        f"No checkpoint found for {candidate}"
    )


    print("\nTRAINING PASS")
    print(
        f"Elapsed: {elapsed/60:.1f} min"
    )

    print("\nCheckpoints found:")

    for ckpt in checkpoints:
        print(
            "   ",
            ckpt.relative_to(local_run_dir)
        )


    # ========================================================
    # UPLOAD DIRECTLY TO:
    #
    # model/
    #   result/
    #
    # No seed folder
    # No trial folder
    # ========================================================

    print("\nUPLOADING TO GOOGLE DRIVE...")

    upload_directory_contents(
        local_run_dir,
        DRIVE_RESULT_FOLDERS[candidate]
    )


    # Verify Drive not empty
    uploaded = drive_children(
        DRIVE_RESULT_FOLDERS[candidate]
    )

    assert uploaded, (
        f"Drive verification failed for {candidate}"
    )


    print(
        f"\n✅ {candidate}: "
        f"TRAIN + DRIVE UPLOAD PASS"
    )


# ============================================================
# FINAL VERIFY
# ============================================================

print("\n")
print("=" * 90)
print("FINAL GOOGLE DRIVE VERIFICATION")
print("=" * 90)

all_ok = True

for model, folder_id in DRIVE_RESULT_FOLDERS.items():

    children = drive_children(folder_id)

    print(f"\n{model}/result")

    if not children:
        print("   ❌ EMPTY")
        all_ok = False
        continue

    for item in children:
        print("   ✓", item["name"])


assert all_ok

print("\n" + "=" * 90)
print("COMPLETE")
print("=" * 90)
print("Models : 4")
print("Seed   :", SEED)
print("Epochs : 100 each")
print("Runs   : 4")
print("Test   : NOT USED")
print("Drive  : VERIFIED")
print("=" * 90)

FINAL 4-MODEL / 1-SEED PRECHECK
Git commit: PASS
yolo26n      -> 100 epochs | seed 20260917
yolo26n_p2   -> 100 epochs | seed 20260917
rfdetr_n     -> 100 epochs | seed 20260917
dfine_n      -> 100 epochs | seed 20260917
D-FINE commit: PASS


Drive yolo26n/result: EMPTY / READY
Drive yolo26n_p2/result: EMPTY / READY
Drive rfdetr_n/result: EMPTY / READY
Drive dfine_n/result: EMPTY / READY

PRECHECK: PASS


##########################################################################################
MODEL 1/4 | yolo26n | seed=20260917 | 100 epochs
##########################################################################################

TRAINING START

TRAINING PASS
Elapsed: 3.1 min

Checkpoints found:
    weights/best.pt
    weights/last.pt

UPLOADING TO GOOGLE DRIVE...
   ↑ BoxF1_curve.png
   ↑ BoxPR_curve.png
   ↑ BoxP_curve.png
   ↑ BoxR_curve.png
   ↑ args.yaml
   ↑ confusion_matrix.png
   ↑ confusion_matrix_normalized.png
   ↑ labels.jpg
   ↑ results.csv
   ↑ results.png
   ↑ train_batch0.jpg
   ↑ train_batch1.jpg
   ↑ train_batch2.jpg
   ↑ train_batch3060.jpg
   ↑ train_batch3061.jpg
   ↑ train_batch3062.jpg
   ↑ val_batch0_labels.jpg
   ↑ val_batch0_pred.jpg
   ↑ val_batch1_labels.jpg
   ↑ val_batch1_pred.jpg
   ↑ val_b

HttpError: <HttpError 403 when requesting https://www.googleapis.com/upload/drive/v3/files?fields=id%2Cname&alt=json&uploadType=resumable returned "The user's Drive storage quota has been exceeded.". Details: "[{'message': "The user's Drive storage quota has been exceeded.", 'domain': 'usageLimits', 'reason': 'storageQuotaExceeded'}]">

In [42]:
# ============================================================
# FINAL ROOT-CAUSE DIAGNOSTIC
# Uses existing low-confidence VAL predictions.
#
# NO TRAINING
# NO TEST SET
# ============================================================

from pathlib import Path
from collections import defaultdict
import csv
import json
import math

import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

ROOT = Path("/content/durian_training")
CONTROL = ROOT / "experiment/controlled_experiment"
AUDIT = CONTROL / "capability_audit"

GT_PATH = CONTROL / "coco/annotations/instances_val.json"

MODELS = [
    "yolo26n",
    "yolo26n_p2",
    "rfdetr_n",
    "dfine_n",
]

NAMES = [
    "leaf_algal",
    "leaf_blight",
    "leaf_colletotrichum",
    "leaf_healthy",
    "leaf_phomopsis",
    "leaf_rhizoctonia",
]

THRESHOLDS = [
    0.001,
    0.01,
    0.025,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

IOU_THR = 0.50

OUT = CONTROL / "root_cause_diagnostic"
OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def iou(a, b):

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    inter = iw * ih

    aa = max(0, ax2-ax1) * max(0, ay2-ay1)
    ba = max(0, bx2-bx1) * max(0, by2-by1)

    union = aa + ba - inter

    return inter / union if union > 0 else 0.0


def size_bin(ratio):

    if ratio < 0.01:
        return "TINY_<1%"

    if ratio < 0.02:
        return "SMALL_1-2%"

    if ratio < 0.05:
        return "MEDIUM_2-5%"

    return "LARGE_>=5%"


SIZE_ORDER = [
    "TINY_<1%",
    "SMALL_1-2%",
    "MEDIUM_2-5%",
    "LARGE_>=5%",
]


# ============================================================
# LOAD GT
# ============================================================

gt_doc = json.loads(GT_PATH.read_text())

images = {
    x["id"]: x
    for x in gt_doc["images"]
}

gts = []

for ann in gt_doc["annotations"]:

    img = images[ann["image_id"]]

    x, y, w, h = ann["bbox"]

    ratio = (
        (w * h)
        /
        (img["width"] * img["height"])
    )

    gts.append({
        "annotation_id": ann["id"],
        "image_id": ann["image_id"],
        "class_id": ann["category_id"] - 1,
        "box": (
            x,
            y,
            x + w,
            y + h,
        ),
        "size": size_bin(ratio),
        "area_ratio": ratio,
    })

assert len(gts) == 1516


# ============================================================
# LOAD ALL LOW-CONFIDENCE PREDICTIONS
# ============================================================

predictions = {}

for model in MODELS:

    path = AUDIT / f"{model}_predictions.csv"

    assert path.exists(), f"Missing: {path}"

    rows = []

    with path.open(
        newline="",
        encoding="utf-8"
    ) as f:

        reader = csv.DictReader(f)

        for r in reader:

            rows.append({
                "image_id":
                    int(r["image_id"]),

                "class_id":
                    int(r["class_id"]),

                "confidence":
                    float(r["confidence"]),

                "box": (
                    float(r["x1"]),
                    float(r["y1"]),
                    float(r["x2"]),
                    float(r["y2"]),
                ),
            })

    predictions[model] = rows

    print(
        f"{model:12s}: "
        f"{len(rows):6d} predictions "
        f"(conf >= {min(x['confidence'] for x in rows):.5f})"
    )


# ============================================================
# MATCH FUNCTION
#
# Correct-class matching first.
# Then wrong-class localization matching.
#
# This allows decomposition:
# CORRECT
# WRONG_CLASS
# MISS
# BACKGROUND_FP
# ============================================================

def evaluate_threshold(model, threshold):

    preds = [
        x for x in predictions[model]
        if x["confidence"] >= threshold
    ]

    gt_by_image = defaultdict(list)
    pred_by_image = defaultdict(list)

    for gi, gt in enumerate(gts):
        gt_by_image[gt["image_id"]].append(gi)

    for pi, pred in enumerate(preds):
        pred_by_image[pred["image_id"]].append(pi)

    status = {
        gi: {
            "status": "MISS",
            "pred_class": None,
            "confidence": 0.0,
            "iou": 0.0,
        }
        for gi in range(len(gts))
    }

    used_gt = set()
    used_pred = set()


    # --------------------------------------------------------
    # PASS 1 — SAME CLASS
    # --------------------------------------------------------

    for image_id, pred_ids in pred_by_image.items():

        pred_ids = sorted(
            pred_ids,
            key=lambda i:
                preds[i]["confidence"],
            reverse=True,
        )

        for pi in pred_ids:

            pred = preds[pi]

            best_gt = None
            best_iou = 0

            for gi in gt_by_image[image_id]:

                if gi in used_gt:
                    continue

                gt = gts[gi]

                if (
                    gt["class_id"]
                    != pred["class_id"]
                ):
                    continue

                overlap = iou(
                    pred["box"],
                    gt["box"]
                )

                if overlap > best_iou:
                    best_iou = overlap
                    best_gt = gi

            if (
                best_gt is not None
                and best_iou >= IOU_THR
            ):

                used_gt.add(best_gt)
                used_pred.add(pi)

                status[best_gt] = {
                    "status": "CORRECT",
                    "pred_class":
                        pred["class_id"],
                    "confidence":
                        pred["confidence"],
                    "iou":
                        best_iou,
                }


    # --------------------------------------------------------
    # PASS 2 — WRONG CLASS BUT LOCALIZED
    # --------------------------------------------------------

    for image_id, pred_ids in pred_by_image.items():

        pred_ids = sorted(
            pred_ids,
            key=lambda i:
                preds[i]["confidence"],
            reverse=True,
        )

        for pi in pred_ids:

            if pi in used_pred:
                continue

            pred = preds[pi]

            best_gt = None
            best_iou = 0

            for gi in gt_by_image[image_id]:

                if gi in used_gt:
                    continue

                gt = gts[gi]

                if (
                    gt["class_id"]
                    == pred["class_id"]
                ):
                    continue

                overlap = iou(
                    pred["box"],
                    gt["box"]
                )

                if overlap > best_iou:
                    best_iou = overlap
                    best_gt = gi

            if (
                best_gt is not None
                and best_iou >= IOU_THR
            ):

                used_gt.add(best_gt)
                used_pred.add(pi)

                status[best_gt] = {
                    "status": "WRONG_CLASS",
                    "pred_class":
                        pred["class_id"],
                    "confidence":
                        pred["confidence"],
                    "iou":
                        best_iou,
                }


    # All remaining predictions are background / duplicate FPs
    background_fp = (
        len(preds) - len(used_pred)
    )

    correct = sum(
        x["status"] == "CORRECT"
        for x in status.values()
    )

    wrong = sum(
        x["status"] == "WRONG_CLASS"
        for x in status.values()
    )

    miss = sum(
        x["status"] == "MISS"
        for x in status.values()
    )

    total_pred = len(preds)

    precision = (
        correct / total_pred
        if total_pred else 0
    )

    recall = (
        correct / len(gts)
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0
    )

    localization_rate = (
        (correct + wrong)
        / len(gts)
    )


    # --------------------------------------------------------
    # PER CLASS
    # --------------------------------------------------------

    per_class = []

    for cls in range(len(NAMES)):

        cls_gt_ids = [
            i for i, gt in enumerate(gts)
            if gt["class_id"] == cls
        ]

        cls_correct = sum(
            status[i]["status"] == "CORRECT"
            for i in cls_gt_ids
        )

        cls_wrong = sum(
            status[i]["status"] == "WRONG_CLASS"
            for i in cls_gt_ids
        )

        cls_miss = sum(
            status[i]["status"] == "MISS"
            for i in cls_gt_ids
        )

        cls_pred_count = sum(
            p["class_id"] == cls
            for p in preds
        )

        cp = (
            cls_correct / cls_pred_count
            if cls_pred_count else 0
        )

        cr = (
            cls_correct / len(cls_gt_ids)
            if cls_gt_ids else 0
        )

        cf1 = (
            2 * cp * cr / (cp + cr)
            if cp + cr else 0
        )

        per_class.append({
            "class_id": cls,
            "class": NAMES[cls],
            "precision": cp,
            "recall": cr,
            "F1": cf1,
            "correct": cls_correct,
            "wrong_class": cls_wrong,
            "miss": cls_miss,
            "GT": len(cls_gt_ids),
            "predictions": cls_pred_count,
        })


    # --------------------------------------------------------
    # SIZE
    # --------------------------------------------------------

    per_size = []

    for size in SIZE_ORDER:

        ids = [
            i for i, gt in enumerate(gts)
            if gt["size"] == size
        ]

        c = sum(
            status[i]["status"] == "CORRECT"
            for i in ids
        )

        w = sum(
            status[i]["status"] == "WRONG_CLASS"
            for i in ids
        )

        m = sum(
            status[i]["status"] == "MISS"
            for i in ids
        )

        per_size.append({
            "size": size,
            "GT": len(ids),
            "correct": c,
            "wrong_class": w,
            "miss": m,
            "recall":
                c / len(ids),
            "localization_rate":
                (c+w) / len(ids),
        })


    return {
        "preds": preds,
        "status": status,
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "correct": correct,
        "wrong_class": wrong,
        "miss": miss,
        "background_fp": background_fp,
        "total_predictions": total_pred,
        "localization_rate": localization_rate,
        "per_class": per_class,
        "per_size": per_size,
    }


# ============================================================
# THRESHOLD SWEEP
# ============================================================

threshold_rows = []
class_threshold_rows = []
size_threshold_rows = []

cache = {}

print("\nRunning threshold sweep...")

for model in MODELS:

    for threshold in THRESHOLDS:

        result = evaluate_threshold(
            model,
            threshold
        )

        cache[(model, threshold)] = result

        threshold_rows.append({
            "model": model,
            "threshold": threshold,
            "precision": result["precision"],
            "recall": result["recall"],
            "F1": result["F1"],
            "localization_rate":
                result["localization_rate"],
            "correct":
                result["correct"],
            "wrong_class":
                result["wrong_class"],
            "miss":
                result["miss"],
            "background_fp":
                result["background_fp"],
            "predictions":
                result["total_predictions"],
        })

        for row in result["per_class"]:

            class_threshold_rows.append({
                "model": model,
                "threshold": threshold,
                **row,
            })

        for row in result["per_size"]:

            size_threshold_rows.append({
                "model": model,
                "threshold": threshold,
                **row,
            })


thr_df = pd.DataFrame(threshold_rows)
cls_thr_df = pd.DataFrame(class_threshold_rows)
size_thr_df = pd.DataFrame(size_threshold_rows)


# ============================================================
# BEST GLOBAL THRESHOLD BY F1
# ============================================================

best_global = (
    thr_df
    .sort_values(
        ["model", "F1"],
        ascending=[True, False]
    )
    .groupby("model")
    .first()
    .reset_index()
)


# ============================================================
# BEST CLASS-SPECIFIC THRESHOLD
# ============================================================

best_class = (
    cls_thr_df
    .sort_values(
        ["model", "class", "F1"],
        ascending=[True, True, False]
    )
    .groupby(
        ["model", "class"]
    )
    .first()
    .reset_index()
)


# ============================================================
# CONFUSION MATRIX AT EACH MODEL'S BEST GLOBAL THRESHOLD
# ============================================================

confusion_rows = []

for _, best in best_global.iterrows():

    model = best["model"]
    threshold = float(best["threshold"])

    result = cache[(model, threshold)]

    matrix = [
        [0 for _ in NAMES]
        for _ in NAMES
    ]

    correct = defaultdict(int)
    wrong = defaultdict(int)
    miss = defaultdict(int)

    for gi, gt in enumerate(gts):

        s = result["status"][gi]

        true_cls = gt["class_id"]

        if s["status"] == "CORRECT":

            matrix[
                true_cls
            ][
                s["pred_class"]
            ] += 1

            correct[true_cls] += 1

        elif s["status"] == "WRONG_CLASS":

            matrix[
                true_cls
            ][
                s["pred_class"]
            ] += 1

            wrong[true_cls] += 1

        else:
            miss[true_cls] += 1


    for true_cls in range(6):

        for pred_cls in range(6):

            confusion_rows.append({
                "model": model,
                "threshold": threshold,
                "true_class":
                    NAMES[true_cls],
                "predicted_class":
                    NAMES[pred_cls],
                "count":
                    matrix[true_cls][pred_cls],
            })


confusion_df = pd.DataFrame(
    confusion_rows
)


# ============================================================
# TOP WRONG-CLASS PATHWAYS
# ============================================================

wrong_paths = (
    confusion_df[
        confusion_df["true_class"]
        !=
        confusion_df["predicted_class"]
    ]
    .sort_values(
        ["model", "count"],
        ascending=[True, False]
    )
)

wrong_paths = (
    wrong_paths[
        wrong_paths["count"] > 0
    ]
)


# ============================================================
# P2 SURVIVAL CHECK
# ============================================================

p2_survival_rows = []

for model in [
    "yolo26n",
    "yolo26n_p2",
]:

    base = len(
        cache[
            (model, 0.001)
        ]["preds"]
    )

    for t in THRESHOLDS:

        result = cache[(model, t)]

        surviving = (
            result["total_predictions"]
        )

        p2_survival_rows.append({
            "model": model,
            "threshold": t,
            "predictions": surviving,
            "survival_fraction":
                surviving / base
                if base else 0,
            "recall":
                result["recall"],
            "tiny_recall":
                next(
                    x["recall"]
                    for x in result["per_size"]
                    if x["size"]
                    == "TINY_<1%"
                ),
            "small_recall":
                next(
                    x["recall"]
                    for x in result["per_size"]
                    if x["size"]
                    == "SMALL_1-2%"
                ),
        })


p2_survival_df = pd.DataFrame(
    p2_survival_rows
)


# ============================================================
# SCORE DISTRIBUTIONS
# ============================================================

score_rows = []

for model in MODELS:

    confs = sorted(
        p["confidence"]
        for p in predictions[model]
    )

    s = pd.Series(confs)

    score_rows.append({
        "model": model,
        "predictions": len(confs),
        "mean": s.mean(),
        "median": s.median(),
        "p10": s.quantile(0.10),
        "p25": s.quantile(0.25),
        "p50": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
    })


score_df = pd.DataFrame(score_rows)


# ============================================================
# ROOT-CAUSE CLASSIFICATION
# ============================================================

diagnosis_rows = []

for model in MODELS:

    b = best_global[
        best_global["model"] == model
    ].iloc[0]

    r001 = cache[(model, 0.001)]
    r025 = cache[(model, 0.25)]

    tiny001 = next(
        x["recall"]
        for x in r001["per_size"]
        if x["size"] == "TINY_<1%"
    )

    tiny025 = next(
        x["recall"]
        for x in r025["per_size"]
        if x["size"] == "TINY_<1%"
    )

    low_localization = (
        r001["localization_rate"] < 0.50
    )

    calibration_gain = (
        b["F1"]
        -
        r025["F1"]
    )

    wrong_fraction = (
        r001["wrong_class"]
        /
        max(
            1,
            r001["correct"]
            +
            r001["wrong_class"]
        )
    )

    causes = []

    if calibration_gain > 0.03:
        causes.append(
            "CALIBRATION ISSUE"
        )

    if low_localization:
        causes.append(
            "LOCALIZATION / LEARNING ISSUE"
        )

    if wrong_fraction > 0.25:
        causes.append(
            "CLASSIFICATION ISSUE"
        )

    if (
        tiny001 < 0.20
    ):
        causes.append(
            "SMALL-OBJECT LEARNING ISSUE"
        )

    if not causes:
        causes.append(
            "NO SINGLE DOMINANT FAILURE"
        )

    diagnosis_rows.append({
        "model": model,

        "best_threshold":
            b["threshold"],

        "F1_at_0.25":
            r025["F1"],

        "best_F1":
            b["F1"],

        "F1_gain_from_calibration":
            calibration_gain,

        "recall_at_0.001":
            r001["recall"],

        "recall_at_0.25":
            r025["recall"],

        "tiny_recall_0.001":
            tiny001,

        "tiny_recall_0.25":
            tiny025,

        "localization_rate_0.001":
            r001["localization_rate"],

        "wrong_class_fraction_of_localized":
            wrong_fraction,

        "evidence":
            " + ".join(causes),
    })


diagnosis_df = pd.DataFrame(
    diagnosis_rows
)


# ============================================================
# SAVE EVERYTHING
# ============================================================

thr_df.to_csv(
    OUT / "confidence_sweep.csv",
    index=False
)

best_global.to_csv(
    OUT / "best_global_threshold.csv",
    index=False
)

best_class.to_csv(
    OUT / "best_class_threshold.csv",
    index=False
)

size_thr_df.to_csv(
    OUT / "size_threshold_sweep.csv",
    index=False
)

confusion_df.to_csv(
    OUT / "confusion_matrix_long.csv",
    index=False
)

wrong_paths.to_csv(
    OUT / "wrong_class_pathways.csv",
    index=False
)

p2_survival_df.to_csv(
    OUT / "p2_survival.csv",
    index=False
)

score_df.to_csv(
    OUT / "score_distribution.csv",
    index=False
)

diagnosis_df.to_csv(
    OUT / "root_cause_summary.csv",
    index=False
)


# ============================================================
# PLOTS
# ============================================================

# F1 vs threshold
plt.figure(figsize=(10,6))

for model in MODELS:

    d = thr_df[
        thr_df["model"] == model
    ]

    plt.plot(
        d["threshold"],
        d["F1"],
        marker="o",
        label=model,
    )

plt.xlabel("Confidence threshold")
plt.ylabel("F1")
plt.title(
    "Validation F1 vs Confidence Threshold"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUT / "f1_threshold.png",
    dpi=180
)

plt.close()


# Recall vs threshold
plt.figure(figsize=(10,6))

for model in MODELS:

    d = thr_df[
        thr_df["model"] == model
    ]

    plt.plot(
        d["threshold"],
        d["recall"],
        marker="o",
        label=model,
    )

plt.xlabel("Confidence threshold")
plt.ylabel("Recall")
plt.title(
    "Validation Recall vs Confidence Threshold"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUT / "recall_threshold.png",
    dpi=180
)

plt.close()


# Tiny recall
plt.figure(figsize=(10,6))

for model in MODELS:

    d = size_thr_df[
        (size_thr_df["model"] == model)
        &
        (
            size_thr_df["size"]
            == "TINY_<1%"
        )
    ]

    plt.plot(
        d["threshold"],
        d["recall"],
        marker="o",
        label=model,
    )

plt.xlabel("Confidence threshold")
plt.ylabel("Tiny-object Recall")
plt.title(
    "Tiny Object Recall vs Confidence Threshold"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    OUT / "tiny_recall_threshold.png",
    dpi=180
)

plt.close()


# ============================================================
# PRINT FINAL DIAGNOSTICS
# ============================================================

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)


print("\n")
print("=" * 120)
print("BEST GLOBAL CONFIDENCE THRESHOLD")
print("=" * 120)

print(
    best_global[
        [
            "model",
            "threshold",
            "precision",
            "recall",
            "F1",
            "localization_rate",
            "wrong_class",
            "background_fp",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


print("\n")
print("=" * 120)
print("BEST PER-CLASS THRESHOLD")
print("=" * 120)

print(
    best_class[
        [
            "model",
            "class",
            "threshold",
            "precision",
            "recall",
            "F1",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


print("\n")
print("=" * 120)
print("TOP WRONG-CLASS PATHWAYS")
print("=" * 120)

for model in MODELS:

    print("\n", model.upper())

    d = (
        wrong_paths[
            wrong_paths["model"] == model
        ]
        .head(12)
    )

    print(
        d[
            [
                "true_class",
                "predicted_class",
                "count",
            ]
        ].to_string(
            index=False
        )
    )


print("\n")
print("=" * 120)
print("YOLO26n vs YOLO26n-P2 LOW-CONFIDENCE SURVIVAL")
print("=" * 120)

print(
    p2_survival_df[
        p2_survival_df[
            "threshold"
        ].isin(
            [
                0.001,
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
            ]
        )
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


print("\n")
print("=" * 120)
print("PREDICTION SCORE DISTRIBUTION")
print("=" * 120)

print(
    score_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


print("\n")
print("=" * 120)
print("ROOT-CAUSE EVIDENCE SUMMARY")
print("=" * 120)

print(
    diagnosis_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)


print("\n")
print("=" * 120)
print("FINAL ROOT-CAUSE DIAGNOSTIC COMPLETE")
print("=" * 120)

print("Validation images :", len(gt_doc["images"]))
print("Validation GT     :", len(gts))
print("Thresholds tested :", len(THRESHOLDS))
print("Test set          : NOT USED")

print("\nSaved to:")
print(OUT)

print("""
Files:
 confidence_sweep.csv
 best_global_threshold.csv
 best_class_threshold.csv
 size_threshold_sweep.csv
 confusion_matrix_long.csv
 wrong_class_pathways.csv
 p2_survival.csv
 score_distribution.csv
 root_cause_summary.csv
 f1_threshold.png
 recall_threshold.png
 tiny_recall_threshold.png
""")

print(
    "PASS — FINAL ROOT-CAUSE DIAGNOSTIC COMPLETE"
)

yolo26n     :  17824 predictions (conf >= 0.00100)
yolo26n_p2  :  18600 predictions (conf >= 0.00181)
rfdetr_n    :  18600 predictions (conf >= 0.05179)
dfine_n     :  18600 predictions (conf >= 0.15140)

Running threshold sweep...


BEST GLOBAL CONFIDENCE THRESHOLD
     model  threshold  precision  recall      F1  localization_rate  wrong_class  background_fp
   dfine_n    0.25000    0.06428 0.19657 0.09688            0.35884          246           4092
  rfdetr_n    0.30000    0.31160 0.24274 0.27290            0.39776          235            578
   yolo26n    0.15000    0.27671 0.22889 0.25054            0.39908          258            649
yolo26n_p2    0.10000    0.20647 0.21042 0.20843            0.40963          302            924


BEST PER-CLASS THRESHOLD
     model               class  threshold  precision  recall      F1
   dfine_n          leaf_algal    0.20000    0.10881 0.07865 0.09130
   dfine_n         leaf_blight    0.25000    0.05986 0.38462 0.10360
   dfine_n leaf_col